# HeatShield heat-risk forecasting

This notebook trains a small forecasting pipeline for 30-, 60-, and 120-minute-ahead heat index. It uses a chronological holdout, compares against a persistence baseline, and excludes `heat_index_f` from the input features because it is calculated from temperature and humidity.

The feature builder uses timestamp joins, so the same code works with the current hourly sample and with the production five-minute feed.

In [2]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = Path("../data/sample_tiger_data.csv")
OUTPUT_DIR = Path("../artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["recorded_at"])
df = df.sort_values(["station_id", "recorded_at"]).reset_index(drop=True)
print(f"Rows: {len(df):,}; stations: {df.station_id.nunique()}; time range: {df.recorded_at.min()} to {df.recorded_at.max()}")

Rows: 100; stations: 6; time range: 2026-10-03 00:00:00+00:00 to 2026-10-03 16:00:00+00:00


In [3]:
# Feature engineering is timestamp-based and does not use heat_index_f as an input.
BASE_FEATURES = ["temperature_f", "humidity_pct", "wind_speed_mph", "solar_radiation", "uhi_offset_f",
                 "latitude", "longitude"]
OPTIONAL_STATIC_FEATURES = ["hvi", "HVI", "canopy_cover", "impervious_pct", "poverty_rate",
                            "pct_elderly", "distance_to_cooling", "vulnerable_population"]
LAG_MINUTES = {"temperature_f": [5, 15, 30, 60], "humidity_pct": [5, 15, 30]}
HORIZONS = [30, 60, 120]

def make_features(raw: pd.DataFrame, horizon_minutes: int) -> tuple[pd.DataFrame, list[str]]:
    data = raw.copy()
    data["recorded_at"] = pd.to_datetime(data["recorded_at"], utc=True)
    data = data.sort_values(["station_id", "recorded_at"])
    available = [c for c in BASE_FEATURES + OPTIONAL_STATIC_FEATURES if c in data.columns]
    result = data[["recorded_at", "station_id", "heat_index_f"] + available].copy()
    for column, minutes_list in LAG_MINUTES.items():
        if column not in data.columns:
            continue
        for minutes in minutes_list:
            lookup = data[["station_id", "recorded_at", column]].copy()
            lookup["recorded_at"] += pd.Timedelta(minutes=minutes)
            lookup = lookup.rename(columns={column: f"{column}_lag_{minutes}m"})
            result = result.merge(lookup, on=["station_id", "recorded_at"], how="left")
    for minutes in [15, 30]:
        for column in ["temperature_f", "humidity_pct"]:
            lag_column = f"{column}_lag_{minutes}m"
            if column in result and lag_column in result:
                result[f"{column}_change_{minutes}m"] = result[column] - result[lag_column]
    for minutes in [15, 30, 60]:
        rolling = (data.set_index("recorded_at").groupby("station_id")["temperature_f"]
                   .rolling(f"{minutes}min", min_periods=1).mean().reset_index(name=f"rolling_temp_mean_{minutes}m"))
        result = result.merge(rolling, on=["station_id", "recorded_at"], how="left")
    rolling_humidity = (data.set_index("recorded_at").groupby("station_id")["humidity_pct"]
                        .rolling("30min", min_periods=1).mean().reset_index(name="rolling_humidity_mean_30m"))
    result = result.merge(rolling_humidity, on=["station_id", "recorded_at"], how="left")
    result["hour_of_day"] = result.recorded_at.dt.hour
    result["day_of_year"] = result.recorded_at.dt.dayofyear
    target = data[["station_id", "recorded_at", "heat_index_f"]].copy()
    target["recorded_at"] -= pd.Timedelta(minutes=horizon_minutes)
    target = target.rename(columns={"heat_index_f": f"target_hi_{horizon_minutes}m"})
    result = result.merge(target, on=["station_id", "recorded_at"], how="left")
    # The sample is hourly, so use the next available reading only when an exact
    # horizon is unavailable. Live five-minute data uses the exact join above.
    target_name = f"target_hi_{horizon_minutes}m"
    if result[target_name].notna().sum() == 0:
        cadence = data.groupby("station_id")["recorded_at"].diff().dt.total_seconds().median()
        future = data[["station_id", "recorded_at", "heat_index_f"]].rename(columns={"heat_index_f": target_name})
        lookup = result[["station_id", "recorded_at"]].copy()
        lookup["target_time"] = lookup["recorded_at"] + pd.Timedelta(minutes=horizon_minutes)
        lookup = lookup.sort_values("target_time")
        future = future.sort_values("recorded_at")
        fallback = pd.merge_asof(lookup, future, left_on="target_time", right_on="recorded_at",
                                 by="station_id", direction="forward",
                                 tolerance=pd.Timedelta(seconds=max(cadence * 1.5, 60)))
        result[target_name] = fallback[target_name].to_numpy()
    feature_columns = [c for c in result.columns if c not in {"recorded_at", "heat_index_f", f"target_hi_{horizon_minutes}m"}]
    return result, feature_columns

datasets = {h: make_features(df, h) for h in HORIZONS}
for horizon, (prepared, features) in datasets.items():
    print(f"{horizon:>3} min: {prepared[f'target_hi_{horizon}m'].notna().sum():>4} labeled rows, {len(features)} features")

 30 min:   94 labeled rows, 25 features
 60 min:   94 labeled rows, 25 features
120 min:   88 labeled rows, 25 features


In [4]:
# Chronological split: the most recent 20% is never used for training.
def chronological_split(prepared: pd.DataFrame, target: str):
    labeled = prepared.dropna(subset=[target]).sort_values("recorded_at").copy()
    cutoff = labeled.recorded_at.quantile(0.8)
    return labeled[labeled.recorded_at < cutoff], labeled[labeled.recorded_at >= cutoff]

models = {}
metrics = []
predictions = {}

for horizon, (prepared, feature_columns) in datasets.items():
    target = f"target_hi_{horizon}m"
    train, test = chronological_split(prepared, target)
    categorical = ["station_id"]
    numeric = [c for c in feature_columns if c not in categorical]
    preprocess = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), numeric),
        ("station", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical),
    ])
    model = Pipeline([("preprocess", preprocess),
                     ("regressor", HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05,
                                                                  max_leaf_nodes=15, random_state=42))])
    model.fit(train[feature_columns], train[target])
    prediction = model.predict(test[feature_columns])
    persistence = test["heat_index_f"].to_numpy()
    metrics.extend([
        {"horizon_minutes": horizon, "model": "persistence", "mae_f": mean_absolute_error(test[target], persistence),
         "rmse_f": mean_squared_error(test[target], persistence) ** 0.5},
        {"horizon_minutes": horizon, "model": "hist_gradient_boosting", "mae_f": mean_absolute_error(test[target], prediction),
         "rmse_f": mean_squared_error(test[target], prediction) ** 0.5},
    ])
    predictions[horizon] = test[["recorded_at", "station_id", "latitude", "longitude", "heat_index_f"]].assign(
        observed_future_heat_index=test[target].to_numpy(), predicted_future_heat_index=prediction)
    residual_std = float(np.std(test[target].to_numpy() - prediction, ddof=1)) if len(test) > 1 else 1.0
    models[horizon] = {"model": model, "features": feature_columns, "residual_std_f": max(residual_std, 0.5)}

metrics_df = pd.DataFrame(metrics)
display(metrics_df.round(2))

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['temperature_f_lag_5m' 'temperature_f_lag_15m' 'temperature_f_lag_30m'
 'humidity_pct_lag_5m' 'humidity_pct_lag_15m' 'humidity_pct_lag_30m'
 'temperature_f_change_15m' 'humidity_pct_change_15m'
 'temperature_f_change_30m' 'humidity_pct_change_30m']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['temperature_f_lag_5m' 'temperature_f_lag_15m' 'temperature_f_lag_30m'
 'humidity_pct_lag_5m' 'humidity_pct_lag_15m' 'humidity_pct_lag_30m'
 'temperature_f_change_15m' 'humidity_pct_change_15m'
 'temperature_f_change_30m' 'humidity_pct_change_30m']. At least one non-missing value is needed for imputation with strategy='medi

,horizon_minutes,model,mae_f,rmse_f
0,30,persistence,5.90,7.34
1,30,hist_gradient_boosting,4.32,5.27
2,60,persistence,2.32,2.81
3,60,hist_gradient_boosting,2.41,2.68
4,120,persistence,4.57,5.02
5,120,hist_gradient_boosting,6.69,6.83


In [5]:
# Application-facing forecast helper. Pass one station's latest feature row(s) from the live feed.
def forecast_latest(latest_rows: pd.DataFrame) -> pd.DataFrame:
    forecasts = []
    for horizon, bundle in models.items():
        prepared, _ = make_features(latest_rows, horizon)
        # In production, latest_rows should include enough history for the lag and rolling features.
        row = prepared.sort_values("recorded_at").groupby("station_id").tail(1)
        values = bundle["model"].predict(row[bundle["features"]])
        output = row[["recorded_at", "station_id", "latitude", "longitude"]].copy()
        output["horizon_minutes"] = horizon
        output["predicted_heat_index_f"] = values
        uncertainty = bundle["residual_std_f"]
        output["probability_hi_90"] = [0.5 * (1 + math.erf((value - 90) / (uncertainty * math.sqrt(2)))) for value in values]
        output["probability_hi_105"] = [0.5 * (1 + math.erf((value - 105) / (uncertainty * math.sqrt(2)))) for value in values]
        forecasts.append(output)
    return pd.concat(forecasts, ignore_index=True)

display(predictions[60].head())

,recorded_at,station_id,latitude,longitude,heat_index_f,observed_future_heat_index,predicted_future_heat_index
79,2026-10-03 12:00:00+00:00,NORTH_HILLS,35.8364,-78.6433,79.4,76.1,78.696171
45,2026-10-03 12:00:00+00:00,DOWNTOWN_CORE,35.7796,-78.6382,84.6,77.8,79.554005
96,2026-10-03 12:00:00+00:00,UMSTEAD_FOREST,35.8900,-78.7500,72.4,69.1,70.611665
12,2026-10-03 12:00:00+00:00,CENTENNIAL_CAMPUS,35.7688,-78.6750,75.7,72.4,75.351943
28,2026-10-03 12:00:00+00:00,CHAVIS_PARK_SE,35.7712,-78.6271,84.0,77.4,79.554005


In [6]:
# Save compact artifacts for the application and a human-readable evaluation table.
metrics_df.to_csv(OUTPUT_DIR / "forecast_metrics.csv", index=False)
for horizon, bundle in models.items():
    import joblib
    joblib.dump(bundle, OUTPUT_DIR / f"heat_index_model_{horizon}m.joblib")
with open(OUTPUT_DIR / "feature_contract.json", "w") as handle:
    json.dump({str(h): bundle["features"] for h, bundle in models.items()}, handle, indent=2)
print(f"Saved models, feature contract, and metrics to {OUTPUT_DIR.resolve()}")

Saved models, feature contract, and metrics to C:\Users\Lenovo\Documents\projects\hackathon\New folder\MergeConflict\ML_training\artifacts
